# Graph-based methods for forecasting realized covariances

Replication of the covariance paper by Zhang, Pu, Cucuringu and Dong, based on the supplied September 2023 manuscript and checked against the authors' implementation. [Journal article](https://doi.org/10.1093/jjfinec/nbae026) · [Authors' scripts](https://github.com/chaozhang-ox/Graph-based-HAR/tree/751f8f0b400a33e2f2fd01501ffdc42a2315f013)

**Question.** Do networks improve forecasts of the entire realized covariance matrix, including pairwise correlations, and does that translate into lower realized portfolio risk?

![Data, two graphs, and forecast reconstruction](docs/architecture.png)

The saved run uses genuine intraday bars for all 27 stocks in the paper. It reproduces the structure of the study on a different feed and forecast period. Published table values are displayed as a separate reference.

## 1. data and reproducibility

Use Python 3.10 or 3.11 and run from this folder after installing `requirements.txt`. The public loader downloads one Parquet file per stock plus SPY at a pinned revision and caches it in `data/syncster`. The full cache is approximately 0.5 GB. [Syncster minute bars](https://huggingface.co/datasets/syncster-data/historical-market-data/tree/eff0dac55d843b3b83caec819efa213021a3e3e5)

**Why these data.** A daily return or a file of per-stock realized variances cannot recover off-diagonal realized covariance. We use synchronized 1-minute prices so every five-minute return vector contains contemporaneous moves for the same 27 tickers. The paper uses licensed LOBSTER quote data from 2007–2021. The public feed uses trade bars and starts in 2016. There were only 798 jointly usable sessions through June 2021 under our alignment rule. We extend the public window to December 2024 to preserve all 27 tickers and the 1,000-session calibration. It is a test of the method on a different period, not a test of whether the paper's original numerical tables can be recovered.

**Alignment.** Convert UTC to New York time and use only the 09:30–16:00 regular session. Start with the opening bar's open, then use each minute's close. Each ticker must have at least 350 observed closes and a valid 09:30 open. Carry a past close forward for at most ten minutes **within** a session. Keep only dates that pass for all 27 stocks and SPY, thereby filtering shortened sessions. Different assets may lose different dates. Forecast horizons count retained sessions, so a multi-session forecast can span additional exchange sessions when observations are missing. Portfolio returns are open-to-close simple returns on this retained calendar, with no overnight returns or transaction costs.

**Original data option.** Set `SOURCE = "lobster"` to run on licensed minute data supplied by the reader. Put one CSV per ticker, including SPY, in `data/lobster` with columns `ts,open,close`. Timestamps are ISO 8601 UTC or include their offset, and label the start of each minute. The price fields should come from prevailing best-bid/ask midpoints at the corresponding boundaries. This standardized adapter expects minute quotes already prepared from the LOBSTER event feed. CaPiRe also publishes a free, registration-based academic dataset with daily realized covariance matrices, but the saved run does not depend on institutional login. [CaPiRe access](https://capire.stat.unipd.it/download/)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from threadpoolctl import threadpool_limits

from market_data import load_market, minute_panel
from models import BASELINE, GRAPH_MODEL, forecast_grid
from evaluation import METRICS, covariance_losses, forecast_table, portfolio_table

In [2]:
SOURCE = "syncster"
HORIZONS = (1, 5, 22)
LOOKBACK = 1000
BOOTSTRAP_DRAWS = 10_000

periods = {"syncster": ("2016-01-04", "2024-12-31"), "lobster": ("2007-07-01", "2021-07-01")}
start, end = periods[SOURCE]
with threadpool_limits(limits=1):
    panel = load_market(SOURCE, Path("data") / SOURCE, start, end)
print(f"{len(panel.symbols)} stocks · {len(panel.dates):,} shared sessions")
print(f"observed input dates: {panel.dates[0]:%Y-%m-%d} through {panel.dates[-1]:%Y-%m-%d}")
display(panel.session_counts.sort_values().rename("usable sessions before intersection").to_frame().head(8))

27 stocks · 1,647 shared sessions
observed input dates: 2016-01-07 through 2024-12-31


,usable sessions before intersection
HON,2058
MMM,2072
AXP,2096
PG,2177
HD,2183
MRK,2187
TRV,2208
DIS,2213


## 2. forecast design

1. **Measure** the realized covariance with five offset grids of five-minute percent log returns,

$$H_t = \frac{1}{5}\sum_{s=0}^{4}\sum_l r_{t,l,s}r_{t,l,s}^{\top}.$$

Estimate quarticity from the same grids. Each covariance estimate is a positive semidefinite Gram matrix.

2. **Decompose** each daily matrix as $H_t=D_tR_tD_t$. Fit pooled HAR regressions with stock or pair-specific intercepts. Features are lag 1 and the means of lags 2–5 and 6–22. The released scripts use cumulative 1/5/22 averages, which span the same three lag columns for linear OLS. We solve the demeaned least-squares problem with pivoted QR to avoid squaring its condition number.

3. **Build separate graphs.** For variances, compare empty, complete, same-sector and Graphical LASSO graphs. GLASSO uses 5-fold CV on past daily returns. For correlations, compare empty, complete and the line graph of the complete stock graph. Pairs like (AAPL, MSFT) and (AAPL, JPM) share a stock and connect. All 351 pairs are included, independently of which stock edges GLASSO selected.

4. **Fit and reconstruct.** Compare the twelve variance/correlation graph combinations and HAR-Cholesky. Reconstruct $\hat D\hat R\hat D$, replacing nonpositive variance forecasts with the stock's smallest positive training target. Clip correlations to $[-1,1]$ and use statsmodels cov_nearest with method="clipped" and threshold=1e-4 where needed. The explicit correlation eigenvalue floor stabilizes inversions and can affect losses when a repair occurs. This is an eigenvalue-clipping repair, without a claim that it minimizes every matrix-distance criterion. HAR-Cholesky reconstructs from predicted lower-triangular factors.

5. **Evaluate held out.** Refit monthly using 1,000 past feature dates, less the $h-1$ incomplete forward labels. Make direct forecasts for 1, 5 and 22 retained sessions. For multi-session targets, sum the daily covariance matrices and then decompose the sum. The article's weekly and monthly indices $h=4,21$ count future offsets in sums starting at zero, so they describe 5 and 22 sessions. No separate tuning year is used for OLS.


In [3]:
with threadpool_limits(limits=1):
    forecasts = forecast_grid(
        panel, horizons=HORIZONS, lookback=LOOKBACK,
        first_forecast=pd.Timestamp("2011-07-01") if SOURCE == "lobster" else None,
        quarticity=True,
    )
    losses = {h: covariance_losses(result) for h, result in forecasts.items()}

for h, result in forecasts.items():
    assert (result.audit.last_train_target < result.audit.first_forecast).all()
    assert (result.audit.graph_return_through < result.audit.first_forecast).all()
    assert len([model for model in result.predictions if "HARQ" not in model]) == 13
coverage = pd.DataFrame({
    h: {"monthly refits": len(f.audit), "origins": len(f.dates),
        "first forecast": f.dates[0].date(), "last forecast": f.dates[-1].date()}
    for h, f in forecasts.items()
}).T
display(coverage)
display(forecasts[1].audit.iloc[:4])

,monthly refits,origins,first forecast,last forecast
1,30,623,2022-07-01,2024-12-31
5,30,619,2022-07-01,2024-12-23
22,29,602,2022-07-01,2024-11-26


,month,first_forecast,first_train_feature,last_train_feature,last_train_target,graph_return_through,training_labels,test_origins,quarticity_power,intercept_mode,glasso_edges,glasso_alpha,glasso_cv_warnings,penalty_increased,correlation_repairs
0,2022-07,2022-07-01,2016-03-03,2022-06-30,2022-06-30,2022-06-30,1000,20,1.0,series,174,0.387914,1,False,0
1,2022-08,2022-08-01,2016-05-05,2022-07-29,2022-07-29,2022-07-29,1000,23,1.0,series,178,0.396227,3,False,0
2,2022-09,2022-09-01,2016-06-24,2022-08-31,2022-08-31,2022-08-31,1000,21,1.0,series,173,0.402857,3,False,0
3,2022-10,2022-10-03,2016-08-08,2022-09-30,2022-09-30,2022-09-30,1000,21,1.0,series,180,0.414232,1,False,0


### Calendar completeness

The retained data calendar changes the calibration experiment. A 1,000-observation window can span more than four exchange years when shared stock observations are missing. The table below compares the retained panel against SPY sessions passing the same screening rule. SPY is a usable-session reference, so the difference does not count every possible exchange holiday or shortened session. This diagnoses missing panel observations without filling synthetic covariance matrices.


In [4]:
reference_calendar = minute_panel(
    "SPY", SOURCE, Path("data") / SOURCE, start, end
).index
retained_counts = pd.Series(1, index=panel.dates).resample("YE").sum()
reference_counts = pd.Series(1, index=reference_calendar).resample("YE").sum()
calendar_coverage = pd.DataFrame({
    "retained sessions": retained_counts.astype(int),
    "usable SPY sessions": reference_counts.astype(int),
    "missing from shared panel": (reference_counts - retained_counts).astype(int),
})
display(calendar_coverage)
window_audit = forecasts[1].audit[[
    "month", "first_train_feature", "last_train_feature", "first_forecast",
]].copy()
window_audit["elapsed calibration years"] = (
    window_audit.last_train_feature - window_audit.first_train_feature
).dt.days / 365.25
display(window_audit.iloc[[0, -1]].round(2))


,retained sessions,usable SPY sessions,missing from shared panel
date,,,
2016-12-31,169,251,82
2017-12-31,148,249,101
2018-12-31,126,249,123
2019-12-31,98,248,150
2020-12-31,178,248,70
2021-12-31,181,251,70
2022-12-31,250,250,0
2023-12-31,248,248,0
2024-12-31,249,249,0


,month,first_train_feature,last_train_feature,first_forecast,elapsed calibration years
0,2022-07,2016-03-03,2022-06-30,2022-07-01,6.32
29,2024-12,2020-06-18,2024-11-27,2024-12-02,4.44


## 3. forecast losses and model confidence sets

The Euclidean loss uses the unique lower triangle including its diagonal. The Frobenius loss counts both off-diagonal halves. For realized covariance $H$ and forecast $\hat\Sigma$,

$$L_E=\sqrt{\sum_i(H_{ii}-\hat\Sigma_{ii})^2+\sum_{i<j}(H_{ij}-\hat\Sigma_{ij})^2},$$

$$L_F=\sqrt{\sum_i(H_{ii}-\hat\Sigma_{ii})^2+2\sum_{i<j}(H_{ij}-\hat\Sigma_{ij})^2},$$

$$L_Q=\log|\hat\Sigma|+\operatorname{tr}(\hat\Sigma^{-1}H).$$

The absolute matrix QLIKE level, and therefore ratios of levels, depend on covariance units. We retain percent-squared units and omit a ratio if its baseline mean is nonpositive. Rankings use raw losses.

The sequential SQ model confidence set uses 10,000 stationary bootstrap draws, mean block length 2 and a 5% exclusion threshold. The notebook also reports sensitivity to longer bootstrap blocks for overlapping targets. The protocol's example MCS p-values refer to selected published loss columns. We calculate p-values separately for each metric on our sample.


In [5]:
def quarticity_variant(model):
    return "HARQ" in model


main_losses = {
    h: rows.loc[~rows.model.map(quarticity_variant)] for h, rows in losses.items()
}
with threadpool_limits(limits=1):
    scores = {h: forecast_table(rows, draws=BOOTSTRAP_DRAWS) for h, rows in main_losses.items()}
for h, table in scores.items():
    print(f"{h} sessions · observed on {SOURCE}")
    display(table.round(4))

1 sessions · observed on syncster


Euclidean              Frobenius                QLIKE       \
                 ratio rank   MCS p     ratio rank   MCS p   ratio rank   
model                                                                     
HAR-Cholesky    0.9376    1  1.0000    0.9388    1  1.0000  1.0790   13   
HAR-DRD         1.0000   12  0.0003    1.0000   12  0.0003  1.0000    9   
GHAR(-,K)       1.0008   13  0.0002    1.0001   13  0.0002  0.9891    7   
GHAR(-,L)       0.9977   11  0.0002    0.9961   11  0.0002  0.9855    3   
GHAR(S,-)       0.9944    9  0.0006    0.9960   10  0.0004  1.0009   12   
GHAR(S,K)       0.9950   10  0.0003    0.9959    9  0.0002  0.9894    8   
GHAR(S,L)       0.9923    8  0.0004    0.9923    8  0.0003  0.9857    4   
GHAR(K,-)       0.9865    6  0.0023    0.9896    7  0.0007  1.0004   10   
GHAR(K,K)       0.9868    7  0.0013    0.9890    6  0.0007  0.9883    5   
GHAR(K,L)       0.9846    3  0.0023    0.9861    3  0.0008  0.9845    1   
GHAR(GL,-)      0.9855    4  0.0021    0.9889    5  0.0007  1.0008   11   
GHAR(GL,K)      0.9855    5  0.0021    0.9881    4  0.0007  0.9889    6   
GHAR(GL,L)      0.9835    2  0.0023    0.9855    2  0.0008  0.9851    2   

                      
               MCS p  
model                 
HAR-Cholesky  0.0001  
HAR-DRD       0.0001  
GHAR(-,K)     0.2611  
GHAR(-,L)     0.5572  
GHAR(S,-)     0.0001  
GHAR(S,K)     0.1996  
GHAR(S,L)     0.3558  
GHAR(K,-)     0.0001  
GHAR(K,K)     0.2994  
GHAR(K,L)     1.0000  
GHAR(GL,-)    0.0001  
GHAR(GL,K)    0.2303  
GHAR(GL,L)    0.5572

5 sessions · observed on syncster


Euclidean              Frobenius                QLIKE       \
                 ratio rank   MCS p     ratio rank   MCS p   ratio rank   
model                                                                     
HAR-Cholesky    0.8741    1  1.0000    0.8787    1  1.0000  0.9999   11   
HAR-DRD         1.0000   12  0.0001    1.0000   12  0.0001  1.0000   12   
GHAR(-,K)       1.0008   13  0.0001    1.0001   13  0.0001  0.9965    7   
GHAR(-,L)       0.9974   11  0.0001    0.9957   11  0.0001  0.9964    5   
GHAR(S,-)       0.9917    9  0.0001    0.9941   10  0.0001  1.0002   13   
GHAR(S,K)       0.9922   10  0.0001    0.9939    9  0.0001  0.9965    8   
GHAR(S,L)       0.9893    8  0.0001    0.9902    8  0.0001  0.9964    6   
GHAR(K,-)       0.9785    7  0.0001    0.9836    7  0.0001  0.9993   10   
GHAR(K,K)       0.9783    6  0.0001    0.9824    6  0.0001  0.9956    4   
GHAR(K,L)       0.9763    4  0.0001    0.9797    3  0.0001  0.9954    2   
GHAR(GL,-)      0.9764    5  0.0001    0.9821    5  0.0001  0.9991    9   
GHAR(GL,K)      0.9758    3  0.0001    0.9804    4  0.0001  0.9954    3   
GHAR(GL,L)      0.9740    2  0.0001    0.9781    2  0.0001  0.9953    1   

                      
               MCS p  
model                 
HAR-Cholesky  0.0010  
HAR-DRD       0.0001  
GHAR(-,K)     0.0072  
GHAR(-,L)     0.0118  
GHAR(S,-)     0.0001  
GHAR(S,K)     0.0015  
GHAR(S,L)     0.0025  
GHAR(K,-)     0.0001  
GHAR(K,K)     0.1340  
GHAR(K,L)     0.2755  
GHAR(GL,-)    0.0001  
GHAR(GL,K)    0.1869  
GHAR(GL,L)    1.0000

22 sessions · observed on syncster


Euclidean              Frobenius                QLIKE       \
                 ratio rank   MCS p     ratio rank   MCS p   ratio rank   
model                                                                     
HAR-Cholesky    0.8086    1  1.0000    0.8143    1  1.0000  0.9956    1   
HAR-DRD         1.0000    5  0.0001    1.0000    4  0.0001  1.0000   11   
GHAR(-,K)       1.0060   12  0.0001    1.0073   11  0.0001  0.9989    4   
GHAR(-,L)       1.0050   10  0.0001    1.0059   10  0.0001  0.9990    6   
GHAR(S,-)       1.0003    7  0.0001    1.0024    5  0.0001  1.0001   13   
GHAR(S,K)       1.0063   13  0.0001    1.0095   13  0.0001  0.9990    7   
GHAR(S,L)       1.0056   11  0.0001    1.0086   12  0.0001  0.9991    9   
GHAR(K,-)       0.9956    3  0.0001    0.9990    3  0.0001  1.0001   12   
GHAR(K,K)       1.0013    9  0.0001    1.0059    9  0.0001  0.9989    5   
GHAR(K,L)       1.0010    8  0.0001    1.0054    8  0.0001  0.9990    8   
GHAR(GL,-)      0.9948    2  0.0001    0.9985    2  0.0001  0.9999   10   
GHAR(GL,K)      1.0000    6  0.0001    1.0047    7  0.0001  0.9988    2   
GHAR(GL,L)      0.9999    4  0.0001    1.0045    6  0.0001  0.9989    3   

                      
               MCS p  
model                 
HAR-Cholesky  1.0000  
HAR-DRD       0.0001  
GHAR(-,K)     0.0003  
GHAR(-,L)     0.0003  
GHAR(S,-)     0.0001  
GHAR(S,K)     0.0003  
GHAR(S,L)     0.0002  
GHAR(K,-)     0.0001  
GHAR(K,K)     0.0003  
GHAR(K,L)     0.0003  
GHAR(GL,-)    0.0001  
GHAR(GL,K)    0.0003  
GHAR(GL,L)    0.0003

In [6]:
reference = pd.DataFrame(
    {
        (1, "HAR-Cholesky"): [1.031, 1.032, 1.071],
        (1, "HAR-DRD"): [1, 1, 1],
        (1, "GHAR(S,L)"): [.982, .981, .985],
        (1, "GHAR(K,L)"): [.975, .975, .982],
        (1, "GHAR(GL,L)"): [.975, .975, .982],
        (5, "GHAR(GL,L)"): [.974, .975, .997],
        (22, "GHAR(GL,L)"): [.991, .991, 1.001],
    }, index=METRICS,
).T.rename_axis(index=["sessions", "model"])
print("selected published reference ratios · Tables 4 and 6 · licensed LOBSTER")
display(reference)
graph_ratios = pd.DataFrame({
    f"{h} sessions": table.xs("ratio", axis=1, level=1).loc[GRAPH_MODEL]
    for h, table in scores.items()
})
fig = px.imshow(
    graph_ratios, text_auto=".3f", aspect="auto", color_continuous_scale="RdBu_r",
    color_continuous_midpoint=1, title="GLASSO + line graph · loss relative to HAR-DRD",
    labels={"color": "loss ratio"},
)
fig.update_layout(template="plotly_white", height=380)
fig.show(renderer="plotly_mimetype")

selected published reference ratios · Tables 4 and 6 · licensed LOBSTER


Euclidean  Frobenius  QLIKE
sessions model                                    
1        HAR-Cholesky      1.031      1.032  1.071
         HAR-DRD           1.000      1.000  1.000
         GHAR(S,L)         0.982      0.981  0.985
         GHAR(K,L)         0.975      0.975  0.982
         GHAR(GL,L)        0.975      0.975  0.982
5        GHAR(GL,L)        0.974      0.975  0.997
22       GHAR(GL,L)        0.991      0.991  1.001

In [7]:
with threadpool_limits(limits=1):
    for h in (5, 22):
        sensitivity = forecast_table(main_losses[h], draws=BOOTSTRAP_DRAWS, block=h)
        included = {
            metric: ", ".join(sensitivity.index[sensitivity[metric]["MCS p"] >= .05])
            for metric in METRICS
        }
        display(pd.Series(included, name=f"{h} sessions · MCS with block length {h}"))

Euclidean                                    HAR-Cholesky
Frobenius                                    HAR-Cholesky
QLIKE        GHAR(K,K), GHAR(K,L), GHAR(GL,K), GHAR(GL,L)
Name: 5 sessions · MCS with block length 5, dtype: object

Euclidean                                         HAR-Cholesky
Frobenius                                         HAR-Cholesky
QLIKE        HAR-Cholesky, GHAR(-,K), GHAR(-,L), GHAR(S,K),...
Name: 22 sessions · MCS with block length 22, dtype: object

## 4. portfolio exercise

Compare equal weights with unconstrained GMVP and long-only GMVP+. We solve the unconstrained linear system and use CVXPY with the OSQP solver for simplex-constrained weights. Risk is annualized percentage standard deviation of realized open-to-close portfolio returns. Turnover is measured after weights drift with realized returns.

The one-session exercise follows the protocol's risk and turnover formulas. Longer horizons summarize separate cohorts of nonoverlapping holding periods, compounded from retained open-to-close returns. Because a retained session can follow a data gap, the horizon is measured in observed panel sessions. These exercises exclude overnight returns, missing-date returns and transaction costs.


In [8]:
with threadpool_limits(limits=1):
    portfolios = {
        h: portfolio_table(
            result, panel.portfolio_return, horizon=h,
            selected=list(scores[1].index) if h == 1 else ["HAR-Cholesky", BASELINE, GRAPH_MODEL],
        )
        for h, result in forecasts.items()
    }
display(portfolios[1].round(4))
print("published one-day annualized GMVP standard deviation · Table 5")
display(pd.Series({"1/N": 11.890, "HAR-DRD": 9.918, "GHAR(GL,L)": 9.799}, name="%"))
for h in (5, 22):
    print(f"{h} sessions · selected portfolio models")
    display(portfolios[h].round(4))

risk = portfolios[1].loc[portfolios[1].model.isin(["1/N", "HAR-Cholesky", BASELINE, "GHAR(K,L)", GRAPH_MODEL])]
fig = px.scatter(
    risk, x="turnover", y="annual risk (%)", hover_name="model", color="portfolio",
    title="One-session portfolios · realized risk and turnover",
    color_discrete_sequence=["#a85a32", "#14776c", "#5c62a8"],
)
fig.update_traces(marker={"size": 11})
fig.update_layout(template="plotly_white", height=450)
fig.show(renderer="plotly_mimetype")

,model,portfolio,annual risk (%),turnover
0,1/N,equal,11.1723,0.0077
1,HAR-Cholesky,GMVP,9.4166,0.5944
2,HAR-Cholesky,GMVP+,9.3178,0.3570
3,HAR-DRD,GMVP,9.4165,0.6711
4,HAR-DRD,GMVP+,9.3326,0.4368
5,"GHAR(-,K)",GMVP,9.4222,0.6126
6,"GHAR(-,K)",GMVP+,9.3159,0.3851
7,"GHAR(-,L)",GMVP,9.3711,0.6565
8,"GHAR(-,L)",GMVP+,9.3099,0.4357
9,"GHAR(S,-)",GMVP,9.4738,0.6556


published one-day annualized GMVP standard deviation · Table 5


1/N           11.890
HAR-DRD        9.918
GHAR(GL,L)     9.799
Name: %, dtype: float64

5 sessions · selected portfolio models


,model,portfolio,annual risk (%),turnover
0,1/N,equal,10.5899,0.0181
1,HAR-Cholesky,GMVP,9.0898,0.6779
2,HAR-Cholesky,GMVP+,8.9324,0.4489
3,HAR-DRD,GMVP,9.1559,0.8081
4,HAR-DRD,GMVP+,8.9785,0.5445
5,"GHAR(GL,L)",GMVP,9.1343,0.6615
6,"GHAR(GL,L)",GMVP+,8.9728,0.4693


22 sessions · selected portfolio models


,model,portfolio,annual risk (%),turnover
0,1/N,equal,10.5572,0.0393
1,HAR-Cholesky,GMVP,8.1208,0.5575
2,HAR-Cholesky,GMVP+,8.4235,0.3831
3,HAR-DRD,GMVP,8.3086,0.5867
4,HAR-DRD,GMVP+,8.5063,0.4159
5,"GHAR(GL,L)",GMVP,8.1826,0.6002
6,"GHAR(GL,L)",GMVP+,8.3925,0.4370


## 5. robustness and use cases

**SPY stress split.** Retrospectively classify dates above SPY's full-panel 90th percentile as stress. These classifications are never used to fit the models.

**HARQ and GHARQ.** The supplied manuscript's equations 17–18 display the interaction $RQ_{t-1}\circ v_{t-1}$, while the protocol asks for $\sqrt{RQ_{t-1}}\circ v_{t-1}$. We retain the displayed manuscript equation for the main measurement-error check and run the protocol's square-root variant on the same folds and targets. Q denotes quarticity augmentation here.

**Use cases.** Compare pairwise covariance contributions before a portfolio risk review. Compare GMVP and GMVP+ to see whether lower forecast risk also requires more trading. Compare loss by forecast horizon before selecting a rebalance schedule.


In [9]:
stress = pd.Series(panel.spy_variance > np.quantile(panel.spy_variance, .9), index=panel.dates)
one_day = main_losses[1].assign(regime=lambda t: t.date.map(stress).map({True: "stress", False: "calm"}))
for label, sample in one_day.groupby("regime"):
    print(f"{label} · {sample.date.nunique()} origins · ratios and MCS vs HAR-DRD")
    with threadpool_limits(limits=1):
        regime_scores = forecast_table(sample, draws=BOOTSTRAP_DRAWS)
    display(regime_scores.round(4))

q_losses = losses[1].loc[losses[1].model.map(quarticity_variant)]
with threadpool_limits(limits=1):
    q_scores = forecast_table(q_losses, baseline="HARQ-DRD", draws=BOOTSTRAP_DRAWS)
print("measurement error check · relative to HARQ-DRD")
display(q_scores.round(4))

forecast = forecasts[1]
day = int(np.argmax(panel.spy_variance[forecast.positions]))
observed = forecast.actual[day]
prediction = forecast.predictions[GRAPH_MODEL][day]
difference = prediction - observed
n = len(panel.symbols)
max_cov = np.max(np.abs(difference[~np.eye(n, dtype=bool)]))
np.fill_diagonal(difference, np.nan)
dif = go.Heatmap(
    z=difference, x=panel.symbols, y=panel.symbols, zmid=0, zmin=-max_cov, zmax=max_cov,
    colorscale="RdBu_r", colorbar={"title": "%²"},
)
fig = go.Figure(dif)
fig.update_layout(
    template="plotly_white", height=680, xaxis_title="stock", yaxis_title="stock",
    title=f"{forecast.dates[day]:%Y-%m-%d} · GHAR(GL,L) covariance forecast minus realized",
)
fig.show(renderer="plotly_mimetype")

calm · 588 origins · ratios and MCS vs HAR-DRD


Euclidean              Frobenius                QLIKE       \
                 ratio rank   MCS p     ratio rank   MCS p   ratio rank   
model                                                                     
HAR-Cholesky    0.9089    1  1.0000    0.9088    1  1.0000  1.0756   13   
HAR-DRD         1.0000   12  0.0003    1.0000   12  0.0004  1.0000    9   
GHAR(-,K)       1.0028   13  0.0001    1.0025   13  0.0001  0.9890    7   
GHAR(-,L)       0.9990   11  0.0002    0.9975   10  0.0001  0.9853    3   
GHAR(S,-)       0.9942    9  0.0004    0.9960    9  0.0004  1.0010   12   
GHAR(S,K)       0.9968   10  0.0003    0.9982   11  0.0001  0.9893    8   
GHAR(S,L)       0.9934    8  0.0003    0.9938    8  0.0004  0.9856    4   
GHAR(K,-)       0.9855    5  0.0005    0.9891    5  0.0004  1.0004   10   
GHAR(K,K)       0.9878    7  0.0005    0.9908    7  0.0004  0.9881    5   
GHAR(K,L)       0.9849    4  0.0005    0.9871    3  0.0004  0.9843    1   
GHAR(GL,-)      0.9845    3  0.0005    0.9886    4  0.0004  1.0007   11   
GHAR(GL,K)      0.9865    6  0.0005    0.9900    6  0.0004  0.9886    6   
GHAR(GL,L)      0.9839    2  0.0005    0.9866    2  0.0004  0.9848    2   

                      
               MCS p  
model                 
HAR-Cholesky  0.0001  
HAR-DRD       0.0001  
GHAR(-,K)     0.2812  
GHAR(-,L)     0.5873  
GHAR(S,-)     0.0001  
GHAR(S,K)     0.2176  
GHAR(S,L)     0.3346  
GHAR(K,-)     0.0002  
GHAR(K,K)     0.3081  
GHAR(K,L)     1.0000  
GHAR(GL,-)    0.0008  
GHAR(GL,K)    0.2630  
GHAR(GL,L)    0.5873

stress · 35 origins · ratios and MCS vs HAR-DRD


Euclidean              Frobenius                QLIKE       \
                 ratio rank   MCS p     ratio rank   MCS p   ratio rank   
model                                                                     
HAR-Cholesky    1.1159   13  0.0001    1.1079   13  0.0001  1.1231   13   
HAR-DRD         1.0000   12  0.0205    1.0000   12  0.0162  1.0000   10   
GHAR(-,K)       0.9882    7  0.0298    0.9871    7  0.0219  0.9913    7   
GHAR(-,L)       0.9894    8  0.0258    0.9884    8  0.0176  0.9884    3   
GHAR(S,-)       0.9960   11  0.0226    0.9959   11  0.0171  0.9997    9   
GHAR(S,K)       0.9840    5  0.0499    0.9828    5  0.0219  0.9906    5   
GHAR(S,L)       0.9853    6  0.0298    0.9842    6  0.0206  0.9876    1   
GHAR(K,-)       0.9932   10  0.0233    0.9924   10  0.0171  1.0007   11   
GHAR(K,K)       0.9809    3  0.0598    0.9789    2  0.0603  0.9911    6   
GHAR(K,L)       0.9823    4  0.0298    0.9805    4  0.0219  0.9880    2   
GHAR(GL,-)      0.9916    9  0.0258    0.9908    9  0.0177  1.0018   12   
GHAR(GL,K)      0.9793    1  1.0000    0.9774    1  1.0000  0.9921    8   
GHAR(GL,L)      0.9808    2  0.0598    0.9790    3  0.0603  0.9889    4   

                      
               MCS p  
model                 
HAR-Cholesky  0.0002  
HAR-DRD       0.0621  
GHAR(-,K)     0.4564  
GHAR(-,L)     0.9208  
GHAR(S,-)     0.0034  
GHAR(S,K)     0.3182  
GHAR(S,L)     1.0000  
GHAR(K,-)     0.0054  
GHAR(K,K)     0.3621  
GHAR(K,L)     0.9208  
GHAR(GL,-)    0.0140  
GHAR(GL,K)    0.3182  
GHAR(GL,L)    0.8686

measurement error check · relative to HARQ-DRD


Euclidean              Frobenius                QLIKE             
                ratio rank   MCS p     ratio rank   MCS p   ratio rank   MCS p
model                                                                         
HARQ-DRD       1.0000   11  0.0001    1.0000   11  0.0001  1.0000   11  0.0001
GHARQ(-,K)     1.0025   12  0.0001    1.0024   12  0.0001  0.9872    7  0.1580
GHARQ(-,L)     0.9991   10  0.0001    0.9981    9  0.0001  0.9837    3  0.4111
GHARQ(S,-)     0.9963    8  0.0001    0.9975    8  0.0002  1.0004   12  0.0001
GHARQ(S,K)     0.9985    9  0.0001    0.9995   10  0.0001  0.9874    8  0.1307
GHARQ(S,L)     0.9953    7  0.0001    0.9955    7  0.0001  0.9838    4  0.1591
GHARQ(K,-)     0.9896    4  0.0729    0.9923    4  0.2685  0.9996    9  0.0004
GHARQ(K,K)     0.9913    6  0.0002    0.9936    6  0.0002  0.9862    5  0.1591
GHARQ(K,L)     0.9886    2  0.0729    0.9902    2  0.2685  0.9825    1  1.0000
GHARQ(GL,-)    0.9887    3  0.6454    0.9918    3  0.2685  0.9997   10  0.0019
GHARQ(GL,K)    0.9901    5  0.0049    0.9929    5  0.0028  0.9866    6  0.1580
GHARQ(GL,L)    0.9876    1  1.0000    0.9897    1  1.0000  0.9828    2  0.5656

In [10]:
with threadpool_limits(limits=1):
    protocol_result = forecast_grid(
        panel, horizons=(1,), lookback=LOOKBACK,
        first_forecast=forecasts[1].dates[0], quarticity=True, quarticity_power=.5,
    )[1]
    protocol_rows = covariance_losses(protocol_result)
protocol_q = protocol_rows.loc[protocol_rows.model.map(quarticity_variant)]
with threadpool_limits(limits=1):
    protocol_scores = forecast_table(
        protocol_q, baseline="HARQ-DRD", draws=BOOTSTRAP_DRAWS,
    )
print("protocol sensitivity · square-root quarticity · relative to its HARQ-DRD")
display(protocol_scores.round(4))
assert protocol_result.dates.equals(forecasts[1].dates)


protocol sensitivity · square-root quarticity · relative to its HARQ-DRD


Euclidean              Frobenius                QLIKE             
                ratio rank   MCS p     ratio rank   MCS p   ratio rank   MCS p
model                                                                         
HARQ-DRD       1.0000   10  0.0001    1.0000   10  0.0002  1.0000   11  0.0001
GHARQ(-,K)     1.0035   12  0.0001    1.0037   12  0.0001  0.9863    7  0.0812
GHARQ(-,L)     0.9997    9  0.0001    0.9988    9  0.0002  0.9829    3  0.0990
GHARQ(S,-)     0.9974    8  0.0001    0.9984    8  0.0002  1.0002   12  0.0001
GHARQ(S,K)     1.0006   11  0.0001    1.0016   11  0.0001  0.9865    8  0.0661
GHARQ(S,L)     0.9970    7  0.0001    0.9970    7  0.0002  0.9831    4  0.0990
GHARQ(K,-)     0.9908    4  0.0050    0.9935    4  0.0781  0.9984    9  0.0001
GHARQ(K,K)     0.9933    6  0.0001    0.9958    6  0.0002  0.9847    5  0.0990
GHARQ(K,L)     0.9901    3  0.0050    0.9918    2  0.0781  0.9812    1  1.0000
GHARQ(GL,-)    0.9895    2  0.7656    0.9927    3  0.0781  0.9984   10  0.0001
GHARQ(GL,K)    0.9918    5  0.0005    0.9948    5  0.0006  0.9849    6  0.0990
GHARQ(GL,L)    0.9888    1  1.0000    0.9910    1  1.0000  0.9813    2  0.7805

### Intercept specification diagnostic

The supplied equations use a vector of intercepts, whereas the released linear GHAR scripts call a pooled regression with one shared intercept. Refit the one-session experiment with that shared-intercept choice on exactly the same sample. This diagnoses a specification difference without selecting settings to match the published ratios.


In [11]:
with threadpool_limits(limits=1):
    shared_result = forecast_grid(
        panel, horizons=(1,), lookback=LOOKBACK,
        first_forecast=forecasts[1].dates[0], intercept_mode="shared",
    )[1]
    shared_scores = forecast_table(
        covariance_losses(shared_result), draws=BOOTSTRAP_DRAWS,
    )
print("released-script shared intercept · relative to its HAR-DRD")
display(shared_scores.loc[["HAR-Cholesky", BASELINE, "GHAR(K,L)", GRAPH_MODEL]].round(4))
assert shared_result.dates.equals(forecasts[1].dates)


released-script shared intercept · relative to its HAR-DRD


Euclidean              Frobenius                QLIKE       \
                 ratio rank   MCS p     ratio rank   MCS p   ratio rank   
model                                                                     
HAR-Cholesky    0.9481    1  1.0000    0.9469    1  1.0000  1.1469   13   
HAR-DRD         1.0000   12  0.0008    1.0000   12  0.0006  1.0000   12   
GHAR(K,L)       0.9888    4  0.0088    0.9908    3  0.0030  0.9837    1   
GHAR(GL,L)      0.9872    2  0.0088    0.9897    2  0.0030  0.9851    3   

                      
               MCS p  
model                 
HAR-Cholesky  0.0001  
HAR-DRD       0.0001  
GHAR(K,L)     1.0000  
GHAR(GL,L)    0.0639

## 6. comparison with the paper's claims

The quoted losses match Tables 4–6 of the supplied manuscript, but the public sample gives a different ranking and confidence set. Compare the same horizon, metric and portfolio constraint. In particular, the published 9.799% figure is for unrestricted GMVP. Its long-only counterpart is 9.901%.

A reduction relative to HAR-DRD does not establish that the graph model beats every competitor. HAR-Cholesky has lower Euclidean and Frobenius losses in this public sample. The one-session graph model's MCS membership must be read separately for each loss.


In [12]:
comparison = []
published_ratios = {
    1: (.975, .975, .982), 5: (.974, .975, .997), 22: (.991, .991, 1.001),
}
for horizon, published in published_ratios.items():
    for metric, value in zip(METRICS, published):
        comparison.append({
            "sessions": horizon, "loss": metric, "paper ratio": value,
            "public ratio": scores[horizon].loc[GRAPH_MODEL, (metric, "ratio")],
            "public MCS p": scores[horizon].loc[GRAPH_MODEL, (metric, "MCS p")],
        })
claim_check = pd.DataFrame(comparison)
claim_check["in public 5% MCS"] = claim_check["public MCS p"] >= .05
display(claim_check.round(4))
risk_comparison = portfolios[1].loc[
    portfolios[1].model.isin([BASELINE, GRAPH_MODEL])
].copy()
risk_comparison["paper annual risk (%)"] = [
    {("HAR-DRD", "GMVP"): 9.918, ("HAR-DRD", "GMVP+"): 9.936,
     ("GHAR(GL,L)", "GMVP"): 9.799, ("GHAR(GL,L)", "GMVP+"): 9.901}[(m, p)]
    for m, p in zip(risk_comparison.model, risk_comparison.portfolio)
]
display(risk_comparison.round(4))


,sessions,loss,paper ratio,public ratio,public MCS p,in public 5% MCS
0,1,Euclidean,0.975,0.9835,0.0023,False
1,1,Frobenius,0.975,0.9855,0.0008,False
2,1,QLIKE,0.982,0.9851,0.5572,True
3,5,Euclidean,0.974,0.9740,0.0001,False
4,5,Frobenius,0.975,0.9781,0.0001,False
5,5,QLIKE,0.997,0.9953,1.0000,True
6,22,Euclidean,0.991,0.9999,0.0001,False
7,22,Frobenius,0.991,1.0045,0.0001,False
8,22,QLIKE,1.001,0.9989,0.0003,False


,model,portfolio,annual risk (%),turnover,paper annual risk (%)
3,HAR-DRD,GMVP,9.4165,0.6711,9.918
4,HAR-DRD,GMVP+,9.3326,0.4368,9.936
25,"GHAR(GL,L)",GMVP,9.4798,0.6082,9.799
26,"GHAR(GL,L)",GMVP+,9.3852,0.4158,9.901


The public experiment gives descriptive short-horizon improvements over HAR-DRD, but it does not recover the paper's all-metric MCS result or the one-session economic advantage of GHAR(GL,L). Missing calibration dates, a different feed and a different forecast period prevent assigning the discrepancy to a single economic cause. The interpretation that firm shocks diffuse through the market is the paper's proposed explanation for horizon decay, not something this experiment identifies causally.

The main model follows the manuscript's series-specific intercepts. The released linear scripts use a shared intercept; a separate diagnostic checks that specification without choosing it based on closeness to the published ratios. Appendix-only variants and original LOBSTER data remain follow-ups. [Detailed audit](docs/protocol-audit.md)
